In [5]:
!pip install spacy gradio nltk pandas 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 3.8 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.53.0 requires huggingface-hub<1.0,>=0.30.0, but you have huggingface-hub 1.30.0 which is incompatible.
tokenizers 0.21.4 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.30.0 which is incompatible.


In [8]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 4.1 MB/s  0:00:03 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [10]:
import gradio as gr
import spacy
import pandas as pd
from nltk.stem import PorterStemmer
from collections import Counter
from spacy import displacy

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Stemmer
stemmer = PorterStemmer()


# 1. Named Entity Recognition
def named_entities(text):
    doc = nlp(text)

    data = []

    for ent in doc.ents:
        data.append({
            "Entity": ent.text,
            "Label": ent.label_
        })

    if data:
        return pd.DataFrame(data)

    return pd.DataFrame(columns=["Entity", "Label"])


# 2. POS Tagging
def pos_tagging(text):
    doc = nlp(text)

    data = []

    for token in doc:
        if not token.is_space:
            data.append({
                "Token": token.text,
                "POS": token.pos_,
                "Tag": token.tag_
            })

    return pd.DataFrame(data)


# 3. POS Distribution
def pos_distribution(text):
    doc = nlp(text)

    pos_counts = Counter(
        token.pos_
        for token in doc
        if not token.is_space
    )

    data = [
        {
            "POS": pos,
            "Count": count
        }
        for pos, count in pos_counts.items()
    ]

    return pd.DataFrame(data).sort_values(
        by="Count",
        ascending=False
    )


# 4. Lemmatization
def lemmatization(text):
    doc = nlp(text)

    data = []

    for token in doc:
        if not token.is_space:
            data.append({
                "Token": token.text,
                "Lemma": token.lemma_
            })

    return pd.DataFrame(data)


# 5. Stemming
def stemming(text):
    doc = nlp(text)

    data = []

    for token in doc:
        if not token.is_space and token.is_alpha:
            data.append({
                "Token": token.text,
                "Stem": stemmer.stem(token.text)
            })

    return pd.DataFrame(data)


# 6. Morphology
def morphology(text):
    doc = nlp(text)

    data = []

    for token in doc:
        if not token.is_space:
            data.append({
                "Token": token.text,
                "Morphology": str(token.morph)
            })

    return pd.DataFrame(data)


# 7. Dependency Parsing
def dependency_parsing(text):
    doc = nlp(text)

    html = displacy.render(
        doc,
        style="dep",
        jupyter=False
    )

    return html


In [11]:
# DASHBOARD

with gr.Blocks(title="NLP Dashboard") as demo:

    gr.Markdown(
        """
        # NLP Dashboard

        Enter a sentence or paragraph and select an NLP operation.
        """
    )

    text_input = gr.Textbox(
        label="Enter Text",
        placeholder="Enter your sentence here...",
        lines=5
    )

    with gr.Tab("Named Entity Recognition"):
        ner_button = gr.Button("Perform NER")
        ner_output = gr.Dataframe(
            headers=["Entity", "Label"],
            label="Named Entities"
        )

        ner_button.click(
            fn=named_entities,
            inputs=text_input,
            outputs=ner_output
        )

    with gr.Tab("POS Tagging"):
        pos_button = gr.Button("Perform POS Tagging")
        pos_output = gr.Dataframe(
            headers=["Token", "POS", "Tag"],
            label="POS Tags"
        )

        pos_button.click(
            fn=pos_tagging,
            inputs=text_input,
            outputs=pos_output
        )

    with gr.Tab("POS Distribution"):
        distribution_button = gr.Button("Calculate POS Distribution")
        distribution_output = gr.Dataframe(
            headers=["POS", "Count"],
            label="POS Distribution"
        )

        distribution_button.click(
            fn=pos_distribution,
            inputs=text_input,
            outputs=distribution_output
        )

    with gr.Tab("Lemmatization"):
        lemma_button = gr.Button("Perform Lemmatization")
        lemma_output = gr.Dataframe(
            headers=["Token", "Lemma"],
            label="Lemmatization"
        )

        lemma_button.click(
            fn=lemmatization,
            inputs=text_input,
            outputs=lemma_output
        )

    with gr.Tab("Stemming"):
        stem_button = gr.Button("Perform Stemming")
        stem_output = gr.Dataframe(
            headers=["Token", "Stem"],
            label="Stemming"
        )

        stem_button.click(
            fn=stemming,
            inputs=text_input,
            outputs=stem_output
        )

    with gr.Tab("Morphology"):
        morph_button = gr.Button("Analyze Morphology")
        morph_output = gr.Dataframe(
            headers=["Token", "Morphology"],
            label="Morphological Features"
        )

        morph_button.click(
            fn=morphology,
            inputs=text_input,
            outputs=morph_output
        )

    with gr.Tab("Dependency Parsing"):
        dependency_button = gr.Button("Show Dependencies")
        dependency_output = gr.HTML(
            label="Dependency Tree"
        )

        dependency_button.click(
            fn=dependency_parsing,
            inputs=text_input,
            outputs=dependency_output
        )


# Launch dashboard
demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
